# Egyptian Clinic AI Voice Agent — Cleaned & Fixed Notebook (v2)
This is the v1 notebook with a second round of bug fixes applied:

**Critical**
1. **Duplicate `_send_whatsapp_text_to_chat` removed** — the second (simpler) definition was overriding the first, which silently disabled the `@lid` resolution *and* the failed-send logging. The v1 "@lid fix" was dead code; now it actually runs.
2. **Double-booking via time-format mismatch fixed** — `CLINIC_HOURS` is now zero-padded (`09:00` not `9:00`), all times are normalized through `strptime` before storage/comparison, and availability parses stored timestamps properly instead of string-slicing.
3. **`poll_and_process_new_messages` default `limit` raised 1 → 25** — with `limit=1`, two messages arriving in one poll interval meant the older one was lost forever.
4. **@lid phone derivation fixed** — for `@lid` contacts, `chat_id.split("@")[0]` is a privacy LID, *not* a phone number, so lookup failed and duplicate patients got registered. We now resolve the real number first, and the self-heal `UPDATE` matches on `whatsapp_chat_id` too.

**High**
5. **DB-level double-booking guard** — partial `UNIQUE` index on `(doctor_id, scheduled_at)` for non-cancelled appointments + `IntegrityError` catch (the SELECT-then-INSERT race is closed).
6. **`route_message` crash no longer drops the patient's message** — wrapped in try/except with a safe default (`agent` / `complex`).
7. **Full validation in `book_appointment` / `reschedule_appointment`** — date/time format, past dates, valid clinic slot, patient/doctor existence, and `PRAGMA foreign_keys = ON`.
8. **Ownership check** — `cancel_appointment` / `reschedule_appointment` now require `patient_id` and verify the appointment belongs to that patient.

**Medium**
9. All outbound patient messages route through `get_whatsapp_target()` (single source of truth for @lid handling).
10. Arabic matching hardened — exact-normalized match first, substring fallback needs ≥3 chars; `ال` is no longer stripped from patient *names* (was mangling names like الاء).
11. Voice-note temp files are deleted after transcription.
12. `createdAt` millisecond/second ambiguity normalized in the polling baseline.
13. `flag_inactive_patients` counts only completed (`done`) visits — no-shows and future bookings aren't "activity".
14. Money rounded to 2 decimals everywhere (SQLite DECIMAL is really float).
15. All writes use local Cairo timestamps via `now_local()` — SQLite's `CURRENT_TIMESTAMP` is UTC, which shifted monthly analytics boundaries.
16. Mid-conversation messages no longer pay for an extra LLM router call — escalation uses a cheap keyword check instead.


## v3 — one more critical fix
17. **`_resolve_lid_to_real_jid` was resolving @lid contacts to a garbage id.** It read the
    generic `GET /contacts/:contactId` endpoint's `number` field, which for an `@lid`
    contact is just the LID's own digits, not the real phone — confirmed live against
    the running OpenWA instance (`number` stayed `"268525801320485"`, a fake 15-digit
    "phone" that doesn't correspond to any real WhatsApp JID). Every reply sent to a
    patient behind WhatsApp's privacy `@lid` mode was silently failing to deliver, even
    though the log line ("Resolved ... -> ...@c.us") looked successful. Now uses the
    purpose-built `GET /contacts/:contactId/phone` endpoint, which returns the real
    MSISDN (confirmed: `{"phone": "201116524289"}` for the same contact). Also removed
    a dead fallback (`data.get("id", {}).get("_serialized")`) that would have raised
    `AttributeError` had it ever been reached, since `id` is a plain string in the
    Contact schema, not an object.
18. **Test cell (`say("عايز احجز معاد...")`) silently dropped `max_concurrency: 1`** by
    redefining `config` right before the call, undoing FIX 18's whole point for that
    demo run. Restored.

## v4 -- two more fixes (a real @lid contact hit both)
19. **Un-normalized phone numbers broke WhatsApp sends with a 500.** Patients type
    their number however they naturally would -- local Egyptian mobile format
    (`01156646810`, 11 digits, leading 0) -- but a WhatsApp chatId needs the
    international MSISDN form (`201156646810`, country code, no leading 0).
    `f"{phone}@c.us"` built from the raw local-format number isn't a valid
    JID, so `send_whatsapp_confirmation` (and cancel/reschedule) 500'd and the
    patient never got their confirmation, even though the booking itself
    succeeded. Added `_normalize_eg_phone()`, applied at the point of collection
    (`lookup_patient`/`register_patient`) and again defensively at send time
    (`get_whatsapp_target`/`_send_whatsapp_text`, so already-stored bad data
    self-heals too), plus a one-time migration cell to fix existing rows.
20. **`whatsapp_chat_id` self-heal only ever matched by guessing `phone`.** That
    guess silently fails whenever `_resolve_lid_to_real_jid` can't resolve an
    `@lid` (OpenWA's phone-resolution is best-effort and can legitimately
    return null for a contact it's never mapped) -- in that case `phone` is the
    LID's own unresolvable digits, which never matches the patient's real row,
    so `whatsapp_chat_id` never gets linked and every later notification falls
    back to a possibly-unreachable phone-based id. `handle_whatsapp_message_v2`
    now links `whatsapp_chat_id` deterministically, by reading the `patient_id`
    straight out of `lookup_patient`/`register_patient`'s own tool result for
    that turn -- no guessing. The old phone-matching heal is kept as a fallback.
21. **The agent sometimes asked the patient for their phone number anyway.** Even though
    `handle_whatsapp_message_v2` already handed it the WhatsApp-derived number as a "known
    fact," the model was free to judge it untrustworthy (e.g. an @lid fallback number that
    doesn't look like a real MSISDN) and ask the patient to state one instead — exactly the
    opposite of the point of booking over WhatsApp. Two changes: (a) `handle_whatsapp_message_v2`
    now looks the patient up itself, deterministically, by `phone` OR `whatsapp_chat_id`
    *before* ever invoking the graph — if found, the agent is simply told the `patient_id`
    directly and never touches `lookup_patient`/`register_patient` at all; this also means a
    returning patient is recognized even on a brand-new conversation thread (e.g. right after
    a kernel restart, since the DB persists but LangGraph's in-memory thread state doesn't).
    (b) Added a hard SYSTEM_PROMPT rule: never ask the patient for their phone number, full stop.
22. **A transient Groq 429 could make the bot reply to the same WhatsApp message
    twice, looking like it "replied with no new message."** `agent()`'s LLM call
    had no retry of its own — a rate limit there bubbled all the way out to the
    poll loop's outer per-message retry queue (FIX 19), which re-submits the
    WHOLE message as a fresh `graph.invoke()` call on the next poll cycle. If any
    state was already persisted from the failed attempt, that retry could produce
    a second reply for a message the patient only sent once. The retry now
    happens INSIDE the single `graph.invoke()` call (`agent()` uses the same
    `_with_groq_retry` helper the transcription step already had, moved up so
    both can share it) — the outer per-message retry queue still exists as a
    last-resort safety net, but ordinary transient 429s never reach it anymore.
    Also added an explicit `📩 New <type> message from <chat>: <preview>` log
    line right before each message is processed, so it's always obvious which
    real incoming message triggered which reply.
23. **`_resolve_lid_to_real_jid` re-checked the same unresolvable @lid on every
    single message.** OpenWA's phone-resolution endpoint is best-effort: some
    @lid contacts never resolve because WhatsApp's own servers/engine simply
    haven't learned the real number behind them (not something we can force).
    That's expected and harmless, but re-querying it fresh every message meant
    an extra HTTP round trip AND the same "could not resolve" line printed
    every single time that contact wrote in. Added `_lid_resolution_cache`
    (chat_id -> resolved jid or itself), populated once per @lid per kernel
    session — the first message from an unresolvable contact still explains
    why, every message after that is silent and instant.

**v5 (this file)**
- **FIX 24** — reject fake @lid resolutions: OpenWA sometimes echoes the LID's own digits back as the "phone"; a real resolution must be a different number.
- **FIX 25** — candidate-based sending with a working-target cache: resolved JID → raw @lid → {digits}@lid fallback, first success remembered per chat. One bad answer from OpenWA can no longer kill every reply.
- Poll filter now also skips `@newsletter` and `@broadcast` senders.


### Imports

In [ ]:
import os
import re
import json
import base64
import tempfile
import calendar
from datetime import datetime, date, timedelta
from typing import List, Optional, Union, Literal
from pydantic import BaseModel, Field
import requests
from dotenv import load_dotenv
from zoneinfo import ZoneInfo

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from groq import Groq

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

from typing_extensions import TypedDict
from typing import Annotated

load_dotenv(override=True)
CAIRO = ZoneInfo("Africa/Cairo")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

print("Imports ready.")

### LLM

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)          # Layer 2 — full reasoning
llm_mini = ChatGroq(model="openai/gpt-oss-20b", temperature=0)       # Layer 1 — fast/cheap routing
llm_full = llm                                                        # alias used by the graph

print(llm.invoke("Reply with just the word: ready").content)
print(llm_mini.invoke("Reply with just the word: ready").content)

In [ ]:
import time

def _with_groq_retry(fn, attempts: int = 4, base_delay: int = 3):
    """FIX 17 (moved up in v4 as FIX 22): shared retry helper for transient Groq
    errors (429/500/503). Used by both the transcription helpers further below
    AND the agent's own LLM call in agent() -- retrying the LLM call INSIDE a
    single graph.invoke() matters: without it, a 429 bubbles all the way out to
    the poll loop's outer per-message retry (FIX 19), which re-submits the
    WHOLE message fresh on the next poll cycle. If the failed attempt already
    left partial state behind, that outer retry can end up replying to the
    same incoming WhatsApp message twice -- which looks exactly like the bot
    spontaneously sending a reply with no new message from the patient."""
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except Exception as e:
            msg = str(e).lower()
            transient = "429" in msg or "rate limit" in msg or "500" in msg or "503" in msg
            if not transient or attempt == attempts:
                raise
            delay = base_delay * (2 ** (attempt - 1))
            print(f"   ⏳ Groq rate-limited (attempt {attempt}/{attempts}) — retrying in {delay}s…")
            time.sleep(delay)

print("_with_groq_retry ready.")


### DATABASE
All tables in one place. `whatsapp_chat_id` is new — it's what fixes the `@lid` bug (see Agent 4 section for why this matters).

In [ ]:
import psycopg2
import psycopg2.errors

# The SAME Postgres the doctor/secretary dashboards read (DATABASE_URL in .env --
# Supabase transaction pooler). autocommit=True is deliberate and load-bearing:
# psycopg2 otherwise opens an implicit transaction on the FIRST statement of any
# kind (even a SELECT) and holds it until commit -- which, through a transaction
# pooler, pins a pooled server backend "idle in transaction" for however long the
# clinic goes between WhatsApp messages. In autocommit every statement is atomic
# on its own and the partial unique index on appointments stays the double-booking
# guard (exactly the semantics the SQLite version relied on). The one multi-
# statement write (Agent 2's payment + procedure lines) opens an explicit
# BEGIN ... COMMIT around just itself.

def _connect():
    return psycopg2.connect(
        os.environ["DATABASE_URL"],
        connect_timeout=10,
        keepalives=1, keepalives_idle=30, keepalives_interval=10, keepalives_count=3,
    )

conn = _connect()
conn.autocommit = True


def _ensure_conn():
    """The pooler/NAT can silently drop a connection that idled for hours between
    patients. Called at the top of every poll cycle: cheap ping, reconnect if dead."""
    global conn
    try:
        cur = conn.cursor()
        cur.execute("SELECT 1")
        cur.fetchone()
    except (psycopg2.OperationalError, psycopg2.InterfaceError):
        print("   \U0001f50c DB connection lost -- reconnecting...")
        try:
            conn.close()
        except Exception:
            pass
        conn = _connect()
        conn.autocommit = True


def now_cairo():
    """Cairo-aware timestamp for every write. The server session runs in UTC, so a
    naive datetime here would be stored 2-3h off and shift the dashboards' "today"."""
    return datetime.now(CAIRO)


# Read-only sanity check: the dashboard app owns this schema. The notebook NEVER
# runs DDL against it -- if something's missing, fix it on the app side.
_required_tables = {"patients", "doctors", "appointments", "payments", "patient_procedures", "procedures"}
cur = conn.cursor()
cur.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public' AND table_name = ANY(%s)
""", (list(_required_tables),))
_missing = _required_tables - {r[0] for r in cur.fetchall()}
assert not _missing, f"Dashboard tables missing from Postgres: {_missing}"
print("Connected to the dashboard's Postgres -- all 6 tables present.")


In [ ]:
# The dashboard app owns this schema -- the notebook only VERIFIES it (no ALTERs,
# no deletes). These are the columns the agent's own queries depend on.
_required_columns = {
    ("patients", "whatsapp_chat_id"),
    ("patients", "phone"),
    ("appointments", "confirmation_sent"),
    ("appointments", "scheduled_at"),
    ("payments", "final_amount"),
    ("patient_procedures", "payment_id"),
}
cur = conn.cursor()
cur.execute("""
    SELECT table_name, column_name FROM information_schema.columns
    WHERE table_schema = 'public'
""")
_have = set(map(tuple, cur.fetchall()))
_missing = _required_columns - _have
assert not _missing, f"Postgres schema is missing columns the agent needs: {_missing}"
print("Schema check OK -- all agent-required columns exist.")


In [ ]:
cur = conn.cursor()
cur.execute("SELECT id, name, user_id FROM doctors WHERE id = 1")
row = cur.fetchone()
assert row, "No doctor with id=1 in Postgres -- run add_doctor_user_link.py first."
# payments.posted_by_doctor_id is an FK to users.id, NOT doctors.id -- resolve it
# from the doctor row instead of hardcoding (they're both 1 today by coincidence).
DOCTOR_USER_ID = row[2]
print(f"Doctor: id={row[0]}, name={row[1]}, user_id={DOCTOR_USER_ID}")


In [ ]:
# Price catalogue -- idempotent: ON CONFLICT(name) leaves existing dashboard rows
# (and any price edits made there) completely alone; only genuinely new names insert.
DEFAULT_PROCEDURES = [
    ("\u062d\u0634\u0648", 200, "per_tooth"),
    ("\u062a\u0646\u0638\u064a\u0641 \u0627\u0644\u0623\u0633\u0646\u0627\u0646", 150, "per_session"),
    ("\u062a\u0631\u0643\u064a\u0628 \u062a\u0627\u062c", 800, "per_tooth"),
    ("\u062e\u0644\u0639", 150, "per_tooth"),
    ("\u0639\u0644\u0627\u062c \u0639\u0635\u0628", 400, "per_tooth"),
    ("\u062a\u0628\u064a\u064a\u0636 \u0627\u0644\u0623\u0633\u0646\u0627\u0646", 200, "per_session"),
    ("\u0632\u0631\u0627\u0639\u0629 \u0633\u0646\u0629", 600, "per_tooth"),
    ("\u062a\u0642\u0648\u064a\u0645 \u0634\u0647\u0631\u064a", 300, "per_month"),
    ("\u0641\u064a\u0646\u064a\u0631", 1200, "per_tooth"),
]

cur = conn.cursor()
added = 0
for name, price, unit in DEFAULT_PROCEDURES:
    cur.execute(
        "INSERT INTO procedures (name, base_price, unit) VALUES (%s, %s, %s) ON CONFLICT (name) DO NOTHING",
        (name, price, unit),
    )
    added += cur.rowcount
print(f"Seeded {added} new procedure(s).")
cur.execute("SELECT id, name, base_price, unit FROM procedures ORDER BY id")
for row in cur.fetchall():
    print(row)


### WhatsApp setup + sending helpers
**Fix explained:** WhatsApp sometimes gives a contact a privacy-protected `@lid` identifier instead of a normal `{phone}@c.us` JID (this is what happened with Eman's account). Any function that *rebuilds* the chat id from a phone number will silently fail for these contacts.

**v2 FIX 1:** in v1 `_send_whatsapp_text_to_chat` was accidentally defined **twice** — the second, simpler definition overrode the first, so the `@lid` resolver and the failed-send log table never ran. The duplicate is gone; there is now exactly one sender, and it resolves `@lid` and logs failures.

`get_whatsapp_target()` is now the **single** place that decides where to message a patient (FIX 9) — every tool (confirmation, cancellation, reschedule) goes through it.


In [ ]:
OPENWA_URL = os.getenv("OPENWA_URL")
OPENWA_API_KEY = os.getenv("OPENWA_API_KEY")
OPENWA_SESSION_ID = os.getenv("OPENWA_SESSION_ID")

print("OpenWA config loaded:", bool(OPENWA_URL and OPENWA_API_KEY and OPENWA_SESSION_ID))

In [ ]:
def _normalize_eg_phone(phone: str) -> str:
    """Normalize a patient-typed phone number to the international MSISDN format
    WhatsApp JIDs require (e.g. "01156646810" / "001156646810" -> "201156646810").
    Handles any number of pasted-on leading zeros -- the DB had real rows like
    "001116524289" and "0001116524289" that the old 11-digit-only rule missed,
    and every send built from them 500'd. Anything that still doesn't look like
    an Egyptian mobile is returned as-is (better to fail loudly on a real send
    than guess wrong)."""
    digits = re.sub(r"\D", "", phone or "")
    stripped = digits.lstrip("0")
    if stripped.startswith("20") and len(stripped) == 12:
        return stripped
    if stripped.startswith("1") and len(stripped) == 10:
        return "20" + stripped
    return digits


_lid_resolution_cache = {}  # chat_id -> resolved jid, or chat_id itself if unresolvable


def _resolve_lid_to_real_jid(chat_id: str) -> str:
    """If chat_id is an @lid (privacy-protected contact), resolve it to the real
    phone number via OpenWA's dedicated phone-resolution endpoint. Falls back to
    the original chat_id if resolution isn't available or fails.

    v3 FIX: this used to call the generic GET /contacts/:contactId endpoint and
    read its `number` field. For an @lid contact, OpenWA's generic contact
    endpoint returns `number` = the LID's own digits (NOT the real MSISDN) --
    verified live: GET /contacts/268525801320485@lid returned
    {"id": "201116524289@c.us", "number": "268525801320485", ...}. The code was
    reading `number`, so it silently built garbage ids like
    "268525801320485@c.us" (not a real WhatsApp JID) and printed
    "Resolved ... -> ...@c.us" as if it worked -- every reply to a patient behind
    a privacy @lid was actually failing to deliver. The dedicated
    GET /contacts/:contactId/phone endpoint exists specifically for this and
    returns the real MSISDN (verified: {"contactId": "...@lid", "phone": "201116524289"}).
    The old `data.get("id", {}).get("_serialized")` fallback was also dead-wrong --
    `id` is a plain string in the Contact schema, so calling .get() on it would
    have raised AttributeError had it ever been reached.

    Note: this endpoint is best-effort -- OpenWA itself can return phone=null
    when the engine has never mapped this @lid to a real number. That's a real
    dead end for THIS lookup path, and it's not something we can force -- it
    depends entirely on whether WhatsApp's own servers/engine have ever learned
    the real number behind this privacy id. handle_whatsapp_message_v2's
    deterministic self-heal (v3 FIX 20) is what actually recovers a usable
    target in that case, by linking whatsapp_chat_id to the patient_id the
    conversation itself established via lookup_patient/register_patient.

    v4 FIX 23: cache the result (success OR "unresolvable") per chat_id for the
    life of the kernel. Without this, every single message from a contact whose
    @lid will NEVER resolve (OpenWA's answer for a given id doesn't change
    moment to moment) re-hits this endpoint and re-prints the same "could not
    resolve" line every time -- pure noise, plus a wasted HTTP round trip per
    message. Cached once, it's silent (and instant) from the second message on."""
    if not chat_id.endswith("@lid"):
        return chat_id
    if chat_id in _lid_resolution_cache:
        return _lid_resolution_cache[chat_id]
    resolved = chat_id  # default: unresolvable, unless proven otherwise below
    try:
        url = f"{OPENWA_URL}/api/sessions/{OPENWA_SESSION_ID}/contacts/{chat_id}/phone"
        headers = {"X-API-Key": OPENWA_API_KEY}
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.ok:
            phone = resp.json().get("phone")
            # v5 FIX 24: some OpenWA versions/contacts echo the LID's OWN digits
            # back as "phone" (live-verified in your logs: resolved -> same digits
            # @c.us -> guaranteed 500). A real resolution must be a DIFFERENT
            # number than the LID itself.
            if phone and phone not in chat_id:
                resolved = f"{phone}@c.us"
                print(f"   ℹ️ Resolved {chat_id} -> {resolved}")
    except Exception as e:
        print(f"   ⚠️ Could not resolve @lid contact {chat_id}: {e}")
    if resolved == chat_id:
        print(f"   ℹ️ {chat_id} has no known phone mapping yet — will keep using the raw chat id "
              f"(won't re-check again this session).")
    _lid_resolution_cache[chat_id] = resolved
    return resolved


def _log_failed_whatsapp_send(chat_id: str, text: str):
    """Persist failed sends so a silent-failure never just vanishes into the print log.
    (failed_whatsapp_sends is the one notebook-owned table, so CREATE here is fine.)"""
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS failed_whatsapp_sends (
            id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
            chat_id TEXT,
            text TEXT,
            created_at TIMESTAMPTZ DEFAULT now()
        )
    """)
    cur.execute(
        "INSERT INTO failed_whatsapp_sends (chat_id, text) VALUES (%s, %s)",
        (chat_id, text)
    )


# FIX 1: this function existed TWICE in v1 -- the second definition (without the
# @lid resolver and without failure logging) silently overrode this one. There is
# now exactly one definition.
_working_target_cache = {}  # chat_id -> the target that last delivered successfully


def _send_whatsapp_text_to_chat(chat_id: str, text: str) -> bool:
    """v5 FIX 25: candidate-based sender. The old version bet everything on ONE
    target from the resolver -- when OpenWA's phone-resolution returned garbage
    (the LID's own digits), the single send 500'd and the reply was simply lost,
    even though sending to the RAW @lid target is live-verified to work on this
    server (July 13 logs). Now we try candidates in order until one delivers:
      1. whatever target last worked for this chat (cached)
      2. the resolved real JID (only if genuinely different from the LID)
      3. the raw chat_id exactly as WhatsApp gave it
      4. if the chat_id is {digits}@c.us, also try {digits}@lid -- recovers the
         case where a fake @c.us was built from LID digits by a phone fallback.
    First success is cached so later sends to this chat go straight to what works."""
    candidates = []
    cached = _working_target_cache.get(chat_id)
    if cached:
        candidates.append(cached)
    resolved = _resolve_lid_to_real_jid(chat_id)
    if resolved != chat_id:
        candidates.append(resolved)
    candidates.append(chat_id)
    if chat_id.endswith("@c.us"):
        candidates.append(chat_id.split("@")[0] + "@lid")
    # dedupe, keep order
    seen = set()
    candidates = [c for c in candidates if not (c in seen or seen.add(c))]

    url = f"{OPENWA_URL}/api/sessions/{OPENWA_SESSION_ID}/messages/send-text"
    headers = {"X-API-Key": OPENWA_API_KEY, "Content-Type": "application/json"}

    for target in candidates:
        try:
            resp = requests.post(url, headers=headers, json={"chatId": target, "text": text}, timeout=15)
        except requests.RequestException as e:
            print(f"   ❌ Send to {target} failed (network): {e}")
            continue
        if resp.ok:
            _working_target_cache[chat_id] = target
            if target != candidates[0]:
                print(f"   ℹ️ Delivered via fallback target {target}")
            return True
        print(f"   ❌ Send to {target} failed ({resp.status_code})")

    _log_failed_whatsapp_send(chat_id, text)
    return False


def _send_whatsapp_text(phone: str, text: str) -> bool:
    """Convenience wrapper for cases where you only have a known real phone number
    (e.g. manual testing). Internally just builds the standard @c.us chat id."""
    return _send_whatsapp_text_to_chat(f"{_normalize_eg_phone(phone)}@c.us", text)


def get_whatsapp_target(patient_id: int, fallback_phone: str = None) -> str:
    """FIX 9: the ONE place that decides where to message a patient -- prefers their
    real stored whatsapp_chat_id (handles @lid), falls back to {phone}@c.us otherwise.
    All tools (confirmation / cancel / reschedule) route through this.
    v3 FIX: the phone fallback is now normalized -- an un-normalized local-format
    stored phone (e.g. "01156646810") built an invalid JID and 500'd on send."""
    cur = conn.cursor()
    cur.execute("SELECT whatsapp_chat_id, phone FROM patients WHERE id = %s", (patient_id,))
    row = cur.fetchone()
    if not row:
        return f"{_normalize_eg_phone(fallback_phone)}@c.us" if fallback_phone else None
    chat_id, phone = row
    return chat_id if chat_id else f"{_normalize_eg_phone(phone)}@c.us"

print("WhatsApp helpers ready.")


In [ ]:
# v3 migration: normalize already-stored patients.phone values to the
# international MSISDN format (needs _normalize_eg_phone, defined above).
# Safe to re-run -- only touches rows that actually need it, and skips (with a
# warning instead of crashing on a UNIQUE constraint) any normalization that
# would collide with another row's phone -- that means the same person was
# registered twice under two different formats and needs a manual merge.
cur = conn.cursor()
cur.execute("SELECT id, phone FROM patients")
rows = cur.fetchall()
existing_phones = {p for _, p in rows}

fixed, skipped = 0, []
for patient_id, phone in rows:
    normalized = _normalize_eg_phone(phone)
    if normalized == phone:
        continue
    if normalized in existing_phones:
        skipped.append((patient_id, phone, normalized))
        continue
    cur.execute("UPDATE patients SET phone = %s WHERE id = %s", (normalized, patient_id))
    existing_phones.discard(phone)
    existing_phones.add(normalized)
    fixed += 1

print(f"Normalized {fixed} patient phone number(s).")
if skipped:
    print("⚠️ Could not auto-normalize (would collide with an existing patient's phone) -- check these by hand:")
    for patient_id, phone, normalized in skipped:
        print(f"   patient_id={patient_id}, phone={phone!r} -> would become {normalized!r}, but that's already taken")


### Tools
**Fix explained:** every id parameter below used to be typed `Union[str, int]`. Groq's tool-calling schema doesn't reliably support that union — it sometimes collapses it to one type internally, and the model can still emit the other type, which Groq's own validator then rejects with a `BadRequestError` before your code even runs. Locking every id param to plain `str` removes the ambiguity.

In [ ]:
@tool
def lookup_patient(phone: str) -> str:
    """Look up a patient by phone number. Returns patient info if found, or a not-found message."""
    phone = _normalize_eg_phone(phone)  # v3 FIX: match however the patient typed it
    cur = conn.cursor()
    cur.execute("SELECT id, name FROM patients WHERE phone = %s", (phone,))
    row = cur.fetchone()
    if row:
        return f"Found patient: patient_id={row[0]}, name={row[1]}"
    return "No patient found with this phone number."


@tool
def register_patient(name: str, phone: str) -> str:
    """Register a new patient. Use this only after confirming the patient is not already in the system.
    Returns the new patient's id -- use that exact patient_id for any follow-up tool call
    (get_available_slots doesn't need it, but book_appointment, get_patient_appointments,
    reschedule_appointment, cancel_appointment, and send_whatsapp_confirmation all do)."""
    phone = _normalize_eg_phone(phone)  # v3 FIX: store in the format WhatsApp sends actually need
    cur = conn.cursor()
    try:
        cur.execute(
            "INSERT INTO patients (name, phone) VALUES (%s, %s) RETURNING id",
            (name, phone)
        )
        new_patient_id = cur.fetchone()[0]
        return f"Registered new patient '{name}' successfully. patient_id={new_patient_id}, phone={phone}. Use this exact patient_id for any later tool call in this conversation."
    except psycopg2.errors.UniqueViolation:
        # Patient already exists (race or duplicate) -- look them up so the agent still
        # gets a real id. autocommit means the failed INSERT didn't poison anything.
        cur.execute("SELECT id FROM patients WHERE phone = %s", (phone,))
        existing = cur.fetchone()
        if existing:
            return f"A patient with this phone number already exists. patient_id={existing[0]}. Use this exact patient_id."
        return "A patient with this phone number already exists."


print("lookup_patient / register_patient ready.")


In [ ]:
# FIX 2: zero-padded — v1 had "9:00"/"9:30", but bookings got stored as "09:00",
# so booked slots never matched CLINIC_HOURS and still showed as available (= double booking).
CLINIC_HOURS = ["09:00","09:30","10:00","10:30","11:00","11:30","12:00","12:30",
                "13:00","13:30","14:00","14:30","15:00","16:00","17:00"]

# Patients can only book within this many days from today. Enforced HERE (the tool
# layer), not just in the prompt, so a misbehaving model can never book past it.
BOOKING_WINDOW_DAYS = 14

def _parse_id(value: str, field_name: str) -> tuple:
    """Safely parse a tool-provided id. Returns (int_value, None) on success or
    (None, error_message) on failure — so a bad/placeholder id from the model
    becomes a normal tool result the agent can react to, instead of crashing
    the whole message pipeline with an uncaught ValueError."""
    try:
        return int(value), None
    except (TypeError, ValueError):
        return None, (
            f"Invalid {field_name}: '{value}' is not a valid numeric id. "
            f"You must use the exact numeric {field_name} returned earlier in this conversation "
            f"by register_patient, lookup_patient, book_appointment, or get_patient_appointments — "
            f"never a placeholder or made-up value. If you don't have it, call the right lookup tool again."
        )


def _validate_booking_datetime(date_str: str, time_str: str) -> tuple:
    """FIX 2 + FIX 7: one gate for every write of a slot. Normalizes the time to
    zero-padded HH:MM, validates the format, rejects past date-times, and rejects
    times that aren't real clinic slots. Returns (scheduled_at, None) on success
    or (None, error_message) the agent can react to."""
    try:
        d = datetime.strptime(str(date_str), "%Y-%m-%d").date()
    except (TypeError, ValueError):
        return None, "Invalid date format. Please ask the patient for a clear date and use YYYY-MM-DD."
    try:
        t = datetime.strptime(str(time_str), "%H:%M").time()
    except (TypeError, ValueError):
        return None, "Invalid time format. Use 24-hour HH:MM, e.g. 09:30 or 14:00."
    time_norm = t.strftime("%H:%M")  # "9:30" -> "09:30"
    if time_norm not in CLINIC_HOURS:
        return None, f"{time_norm} is not a clinic slot. Valid slots: {', '.join(CLINIC_HOURS)}."
    scheduled_dt = datetime.combine(d, t, tzinfo=CAIRO)
    if scheduled_dt < datetime.now(CAIRO):
        return None, f"That date/time ({d.isoformat()} {time_norm}) is in the past. Please ask the patient for an upcoming date."
    max_day = date_cls_today() + timedelta(days=BOOKING_WINDOW_DAYS)
    if d > max_day:
        return None, (
            f"That date ({d.isoformat()}) is beyond the clinic's booking window. Patients can only book "
            f"within the next {BOOKING_WINDOW_DAYS} days (latest bookable day: {max_day.isoformat()}). "
            f"Politely tell the patient booking only opens two weeks ahead and ask for a nearer date."
        )
    return scheduled_dt, None  # tz-aware datetime -- psycopg2 stores it as correct timestamptz


def _record_exists(table: str, record_id: int) -> bool:
    cur = conn.cursor()
    cur.execute(f"SELECT 1 FROM {table} WHERE id = %s", (record_id,))
    return cur.fetchone() is not None


def date_cls_today():
    return datetime.now(CAIRO).date()


@tool
def get_available_slots(doctor_id: str, date: str) -> str:
    """Get available appointment slots for a doctor on a given date (format: YYYY-MM-DD)."""
    try:
        requested_date = datetime.strptime(date, "%Y-%m-%d").date()
    except ValueError:
        return "Invalid date format. Please ask the patient for a clear date and use YYYY-MM-DD."

    if requested_date < date_cls_today():
        return f"That date ({date}) is in the past. Please ask the patient for a valid upcoming date."

    max_day = date_cls_today() + timedelta(days=BOOKING_WINDOW_DAYS)
    if requested_date > max_day:
        return (f"That date ({date}) is beyond the clinic's booking window. Patients can only book "
                f"within the next {BOOKING_WINDOW_DAYS} days (latest bookable day: {max_day.isoformat()}). "
                f"Politely tell the patient booking only opens two weeks ahead and ask for a nearer date.")

    doctor_id_int, err = _parse_id(doctor_id, "doctor_id")
    if err:
        return err
    if not _record_exists("doctors", doctor_id_int):
        return f"No doctor with doctor_id={doctor_id_int} exists."

    cur = conn.cursor()
    # timestamptz comes back as tz-aware datetimes; compare on the Cairo-local date
    # (the same conversion the dashboard's "today" queries use).
    cur.execute("""
        SELECT scheduled_at FROM appointments
        WHERE doctor_id = %s
          AND (scheduled_at AT TIME ZONE 'Africa/Cairo')::date = %s
          AND status != 'cancelled'
    """, (doctor_id_int, requested_date))
    booked = {row[0].astimezone(CAIRO).strftime("%H:%M") for row in cur.fetchall()}

    available = [t for t in CLINIC_HOURS if t not in booked]
    if requested_date == date_cls_today():
        now_hm = datetime.now(CAIRO).strftime("%H:%M")
        available = [t for t in available if t > now_hm]

    if not available:
        return f"No available slots for doctor {doctor_id_int} on {date}."
    return f"Available slots on {date}: {', '.join(available)}"


@tool
def book_appointment(patient_id: str, doctor_id: str, date: str, time: str) -> str:
    """Book an appointment. date format: YYYY-MM-DD, time format: HH:MM (must be one of the available slots).
    Returns the new appointment's id — use that exact id for any follow-up tool call like send_whatsapp_confirmation."""
    patient_id_int, err = _parse_id(patient_id, "patient_id")
    if err:
        return err
    doctor_id_int, err = _parse_id(doctor_id, "doctor_id")
    if err:
        return err

    # FIX 7: existence checks (SQLite FKs were off in v1 — ghost appointments were possible)
    if not _record_exists("patients", patient_id_int):
        return f"No patient with patient_id={patient_id_int} exists. Call lookup_patient or register_patient first and use the id it returns."
    if not _record_exists("doctors", doctor_id_int):
        return f"No doctor with doctor_id={doctor_id_int} exists."

    scheduled_at, err = _validate_booking_datetime(date, time)
    if err:
        return err

    cur = conn.cursor()
    cur.execute("""
        SELECT id FROM appointments
        WHERE doctor_id = %s AND scheduled_at = %s AND status != 'cancelled'
    """, (doctor_id_int, scheduled_at))
    if cur.fetchone():
        return "That slot was just taken. Please choose another time."
    # FIX 5: the SELECT above can still race — the unique index is the real guard.
    try:
        cur.execute("""
            INSERT INTO appointments (patient_id, doctor_id, scheduled_at, status)
            VALUES (%s, %s, %s, 'confirmed') RETURNING id
        """, (patient_id_int, doctor_id_int, scheduled_at))
        new_appointment_id = cur.fetchone()[0]
    except psycopg2.errors.UniqueViolation:
        return "That slot was just taken. Please choose another time."
    norm_time = scheduled_at.strftime("%H:%M")  # echo the NORMALIZED time back to the agent
    return f"Appointment booked successfully. appointment_id={new_appointment_id}, patient_id={patient_id_int}, doctor_id={doctor_id_int}, date={date}, time={norm_time}."


print("get_available_slots / book_appointment ready.")


In [ ]:
@tool
def get_patient_appointments(patient_id: str) -> str:
    """Get all upcoming (non-cancelled) appointments for a patient, with their ids,
    so the agent can identify which one to reschedule or cancel."""
    patient_id_int, err = _parse_id(patient_id, "patient_id")
    if err:
        return err

    cur = conn.cursor()
    cur.execute("""
        SELECT id, scheduled_at, status
        FROM appointments
        WHERE patient_id = %s AND status != 'cancelled'
        ORDER BY scheduled_at ASC
    """, (patient_id_int,))
    rows = cur.fetchall()
    if not rows:
        return "This patient has no upcoming appointments."
    # timestamptz arrives in UTC -- always show the patient Cairo wall-clock time
    lines = [
        f"appointment_id={r[0]}, date_time={r[1].astimezone(CAIRO).strftime('%Y-%m-%d %H:%M')}, status={r[2]}"
        for r in rows
    ]
    return "Upcoming appointments:\n" + "\n".join(lines)


def _load_owned_appointment(appointment_id: str, patient_id: str) -> tuple:
    """FIX 8: shared loader that enforces ownership — a patient (or a hallucinated id)
    can no longer cancel/reschedule someone else's appointment.
    Returns (row_dict, None) or (None, error_message)."""
    appointment_id_int, err = _parse_id(appointment_id, "appointment_id")
    if err:
        return None, err
    patient_id_int, err = _parse_id(patient_id, "patient_id")
    if err:
        return None, err

    cur = conn.cursor()
    cur.execute("""
        SELECT a.id, a.patient_id, a.doctor_id, a.status, a.scheduled_at,
               p.name, p.phone
        FROM appointments a JOIN patients p ON a.patient_id = p.id
        WHERE a.id = %s
    """, (appointment_id_int,))
    row = cur.fetchone()
    if not row:
        return None, "No appointment found with this id."
    if row[1] != patient_id_int:
        return None, (
            f"appointment_id={appointment_id_int} does not belong to patient_id={patient_id_int}. "
            f"Call get_patient_appointments with the correct patient_id and use one of the ids it returns."
        )
    return {
        "id": row[0], "patient_id": row[1], "doctor_id": row[2],
        "status": row[3], "scheduled_at": row[4], "name": row[5], "phone": row[6],
    }, None


@tool
def cancel_appointment(patient_id: str, appointment_id: str) -> str:
    """Cancel an existing appointment. Requires BOTH the patient_id (from lookup_patient/register_patient)
    and the appointment_id (from get_patient_appointments/book_appointment) — the appointment must belong to that patient.
    Automatically sends a WhatsApp notification to the patient — no need to call any other tool after this."""
    appt, err = _load_owned_appointment(appointment_id, patient_id)
    if err:
        return err
    if appt["status"] == "cancelled":
        return "This appointment is already cancelled."

    cur = conn.cursor()
    cur.execute("UPDATE appointments SET status = 'cancelled' WHERE id = %s", (appt["id"],))

    target = get_whatsapp_target(appt["patient_id"], fallback_phone=appt["phone"])  # FIX 9
    when = appt["scheduled_at"].astimezone(CAIRO).strftime("%Y-%m-%d %H:%M")
    sent = _send_whatsapp_text_to_chat(target, f"Hi {appt['name']}, your appointment on {when} has been cancelled.")
    note = "WhatsApp notification sent." if sent else "WhatsApp notification failed to send."
    return f"Appointment {appt['id']} has been cancelled. {note}"


@tool
def reschedule_appointment(patient_id: str, appointment_id: str, new_date: str, new_time: str) -> str:
    """Reschedule an existing appointment. Requires BOTH the patient_id and the appointment_id —
    the appointment must belong to that patient. new_date format: YYYY-MM-DD, new_time format: HH:MM.
    Automatically sends a WhatsApp notification to the patient — no need to call any other tool after this."""
    appt, err = _load_owned_appointment(appointment_id, patient_id)
    if err:
        return err
    if appt["status"] == "cancelled":
        return "Can't reschedule a cancelled appointment. Please book a new one."

    new_scheduled_at, err = _validate_booking_datetime(new_date, new_time)  # FIX 2 + 7
    if err:
        return err

    cur = conn.cursor()
    cur.execute("""
        SELECT id FROM appointments
        WHERE doctor_id = %s AND scheduled_at = %s AND status != 'cancelled' AND id != %s
    """, (appt["doctor_id"], new_scheduled_at, appt["id"]))
    if cur.fetchone():
        return "That new slot is already taken. Please choose another time."

    try:
        cur.execute("UPDATE appointments SET scheduled_at = %s WHERE id = %s", (new_scheduled_at, appt["id"]))
    except psycopg2.errors.UniqueViolation:  # FIX 5: unique index guards the race here too
        return "That new slot is already taken. Please choose another time."

    target = get_whatsapp_target(appt["patient_id"], fallback_phone=appt["phone"])  # FIX 9
    sent = _send_whatsapp_text_to_chat(target, f"Hi {appt['name']}, your appointment has been moved to {new_date} at {new_time}.")
    note = "WhatsApp notification sent." if sent else "WhatsApp notification failed to send."
    return f"Appointment {appt['id']} rescheduled to {new_date} at {new_time}. {note}"


print("cancel_appointment / reschedule_appointment / get_patient_appointments ready.")


In [ ]:
CLINIC_LATITUDE = 30.005726365904355
CLINIC_LONGITUDE = 31.468899256922267
CLINIC_NAME = "Egyptian Dental Clinic"  # change to your actual clinic name
CLINIC_MAPS_LINK = f"https://www.google.com/maps?q={CLINIC_LATITUDE},{CLINIC_LONGITUDE}"

@tool
def send_whatsapp_confirmation(patient_id: str, appointment_id: str) -> str:
    """Send a WhatsApp booking confirmation and the clinic location to the patient.
    Safe to call multiple times — will not resend if already sent for this appointment."""
    patient_id_int, err = _parse_id(patient_id, "patient_id")
    if err:
        return err
    appointment_id_int, err = _parse_id(appointment_id, "appointment_id")
    if err:
        return err

    cur = conn.cursor()
    cur.execute("""
        SELECT p.name, p.phone, a.scheduled_at, a.status, a.confirmation_sent
        FROM appointments a JOIN patients p ON a.patient_id = p.id
        WHERE a.id = %s AND p.id = %s
    """, (appointment_id_int, patient_id_int))
    row = cur.fetchone()
    if not row:
        return "Could not find this appointment/patient to send confirmation."

    name, phone, scheduled_at, status, confirmation_sent = row

    if confirmation_sent:
        return f"Confirmation was already sent for appointment {appointment_id_int}. No need to send again."

    target = get_whatsapp_target(patient_id_int, fallback_phone=phone)  # FIX 9
    when = scheduled_at.astimezone(CAIRO).strftime("%Y-%m-%d %H:%M")
    resp1_ok = _send_whatsapp_text_to_chat(target, f"Hi {name}, your appointment is confirmed for {when}.")
    resp2_ok = _send_whatsapp_text_to_chat(target, f"📍 {CLINIC_NAME} location:\n{CLINIC_MAPS_LINK}")

    if resp1_ok and resp2_ok:
        cur.execute("UPDATE appointments SET confirmation_sent = TRUE WHERE id = %s", (appointment_id_int,))
        return f"Confirmation and location sent to {name} at {phone}."
    elif resp1_ok:
        return "Confirmation sent, but location message failed."
    elif resp2_ok:
        return "Location sent, but confirmation message failed."
    else:
        return "Both WhatsApp sends failed."


tools = [
    lookup_patient, register_patient, get_available_slots, book_appointment,
    reschedule_appointment, cancel_appointment, send_whatsapp_confirmation,
    get_patient_appointments,
]
print("All", len(tools), "tools ready:", [t.name for t in tools])


### Agent 1 — Voice Receptionist graph (LangGraph, dynamic mini/full model selection)

In [ ]:
SYSTEM_PROMPT = """You are a friendly Egyptian clinic voice receptionist.
Speak in Egyptian Arabic dialect with patients.

HARD RULES — never break these:
- NEVER ask the patient for their phone number, and NEVER ask them to confirm/retype/spell it.
  You are always given it automatically at the start of the conversation, in a system note (their
  WhatsApp number) — or, if they're a returning patient, the system note gives you their patient_id
  directly instead. Use whichever one you're given. This is the whole point of booking over
  WhatsApp: the patient never has to state their own number.
- NEVER invent, guess, or make up an id (patient_id, appointment_id) for any tool call, and NEVER
  write a placeholder like '<patient_id>' or '<appointment_id>' as an argument. The ONLY valid id is
  a real number that appeared earlier in this conversation inside a tool result or a system note, e.g.
  "patient_id=7" from register_patient/lookup_patient, or "appointment_id=12" from book_appointment/
  get_patient_appointments. If you don't have that number yet, call the right lookup tool first
  (lookup_patient or get_patient_appointments) instead of calling book_appointment/reschedule_appointment/
  cancel_appointment/send_whatsapp_confirmation.
- NEVER call register_patient if lookup_patient already found the patient, or if the system note
  already gives you a patient_id — use that patient_id directly.
- NEVER call send_whatsapp_confirmation unless book_appointment just succeeded in this conversation.
- NEVER call get_available_slots when the patient has given you NO date at all — ask for one in plain text first.
  WRONG: patient says "I want to book" (no date given) -> you call get_available_slots with today's date or any guessed date.
  RIGHT: patient says "I want to book" (no date given) -> you reply asking "What date would you like?"
- BUT when the patient DOES give a date in any natural form, resolve it YOURSELF silently using the
  current-date note at the end of this prompt — NEVER ask them for the year, never ask them to rephrase
  a date you can resolve:
  * relative words: بكرة = tomorrow, بعد بكرة = day after tomorrow, "الأربع الجاي"/"next Wednesday" =
    the next occurrence of that weekday strictly after today, "الأسبوع الجاي" = same weekday next week.
  * day-month with no year (like "30-8" or "٥ سبتمبر"): resolve to the NEXT future occurrence of that
    day-month (normally this year; next year only around New Year). Never ask for the year.
- Bookings are ONLY accepted within the next 14 days. If the resolved date falls beyond that, do NOT
  call any tool. In ONE message, in Egyptian Arabic: (a) tell them that date is out of our booking
  range — احنا بنحجز في حدود أسبوعين بس من النهارده, (b) tell them the LATEST bookable day (today + 14
  days, computed from the current-date note), and (c) immediately offer to book them a day INSIDE the
  window instead, e.g. "تحب احجزلك يوم ايه قبل كده؟". Never just refuse and stop — always steer them
  to a bookable day.
- cancel_appointment and reschedule_appointment BOTH require the patient_id AND the appointment_id, and the
  appointment must belong to that patient. Always use the patient_id from the system note or lookup_patient,
  and call get_patient_appointments (for the appointment_id) before cancelling or rescheduling.
- When a patient asks to reschedule or cancel without specifying which appointment or giving a new time, call get_patient_appointments FIRST and mention the existing appointment details back to them before asking what they'd like to change.

Correct step order for booking:
1. Check the system note at the very start of the conversation.
   - If it says the patient is already known and gives you a patient_id: skip straight to step 3
     using that exact patient_id. Do NOT call lookup_patient or register_patient again.
   - Otherwise it gives you their WhatsApp phone number: call lookup_patient with that exact number
     immediately. Never ask the patient for it, and never wait for them to state it themselves.
2. If lookup_patient does NOT find them: ask ONLY for their name — do NOT ask for phone (you already
   have it from the system note), and do NOT ask for age or gender (we don't collect them). Then call
   register_patient(name, phone) — its result includes the real patient_id, use that exact number from
   now on. If FOUND: use the patient_id from lookup_patient's result directly.
3. Ask the patient what they need (booking, reschedule, cancel).
4. Ask for the preferred date in plain text. WAIT for their reply. Once they state a date in ANY form
   (relative, day-month, full), resolve it to YYYY-MM-DD yourself per the rules above and call
   get_available_slots for doctor_id=1 with it.
5. Once the patient confirms a specific time, call book_appointment using the real patient_id from step 1/2. Always pass the time zero-padded in 24-hour HH:MM format (09:30, not 9:30).
6. Only after book_appointment succeeds, call send_whatsapp_confirmation using the real patient_id and the appointment_id book_appointment just returned.
7. Keep responses short and natural, like a real phone conversation.
"""

class ClinicState(TypedDict):
    messages: Annotated[list, add_messages]
    use_full_model: bool

llm_mini_with_tools = llm_mini.bind_tools(tools)
llm_full_with_tools = llm_full.bind_tools(tools)

AR_WEEKDAYS = ["الاثنين", "الثلاثاء", "الأربعاء", "الخميس", "الجمعة", "السبت", "الأحد"]

def _dated_system_prompt() -> str:
    """The system prompt isn't persisted by the checkpointer (agent() only returns the
    response), so it's rebuilt on every call — which lets us stamp it with the CURRENT
    date each turn. That's what makes 'بكرة' and 'الأربع الجاي' resolvable, keeps the
    year implicit, and stays correct even if the kernel runs across midnight/new year."""
    now = datetime.now()
    return SYSTEM_PROMPT + (
        f"\n\nCurrent-date note: today is {now.strftime('%Y-%m-%d')}, "
        f"a {now.strftime('%A')} (يوم {AR_WEEKDAYS[now.weekday()]}). "
        f"Resolve every relative or year-less date the patient gives against this."
    )

def agent(state: ClinicState) -> ClinicState:
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=_dated_system_prompt())] + messages
    use_full_model = state.get("use_full_model", False)
    model_to_use = llm_full_with_tools if use_full_model else llm_mini_with_tools
    # v4 FIX 22: retry a transient Groq failure HERE, inside this single
    # graph.invoke() call, instead of letting it bubble out to the poll loop's
    # outer per-message retry (FIX 19) -- see _with_groq_retry's docstring.
    response = _with_groq_retry(lambda: model_to_use.invoke(messages))
    return {"messages": [response]}

builder = StateGraph(ClinicState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

print("Graph compiled with", len(tools), "tools:", [t.name for t in tools])


In [ ]:
def say(text, use_full_model=False):
    result = graph.invoke(
        {"messages": [HumanMessage(content=text)], "use_full_model": use_full_model},
        config
    )
    print("Patient:", text)
    print("Agent:", result["messages"][-1].content)
    print("---")
    return result

print("say() ready.")

In [ ]:
config = {"configurable": {"thread_id": "call_test_100"}, "max_concurrency": 1}

Try it — pick a fresh phone number so `lookup_patient` doesn't collide with earlier test data:

In [ ]:
# DISABLED: this demo used a FAKE phone number -- against the shared dashboard
# Postgres it would register a junk patient and book real appointments the
# secretary would see. Test through real WhatsApp messages instead.
# config = {"configurable": {"thread_id": "call_test_100"}, "max_concurrency": 1}
# say("عايز احجز معاد، رقمي 201066677788")
print("skipped -- demo disabled (writes to the live dashboard DB).")


In [ ]:
# DISABLED: this demo used a FAKE phone number -- against the shared dashboard
# Postgres it would register a junk patient and book real appointments the
# secretary would see. Test through real WhatsApp messages instead.
# say("محمد سيد، عمري 28، ذكر")
print("skipped -- demo disabled (writes to the live dashboard DB).")


In [ ]:
# DISABLED: this demo used a FAKE phone number -- against the shared dashboard
# Postgres it would register a junk patient and book real appointments the
# secretary would see. Test through real WhatsApp messages instead.
# say("يوم 9 اغسطس 2026")
print("skipped -- demo disabled (writes to the live dashboard DB).")


In [ ]:
# DISABLED: this demo used a FAKE phone number -- against the shared dashboard
# Postgres it would register a junk patient and book real appointments the
# secretary would see. Test through real WhatsApp messages instead.
# say("الساعة 10")
print("skipped -- demo disabled (writes to the live dashboard DB).")


### Agent 2 — Doctor Recording Agent

In [ ]:
class ProcedureItem(BaseModel):
    name: str = Field(description="Procedure name exactly as the doctor said it, in Egyptian Arabic")
    quantity: int = Field(default=1, description="How many teeth/sessions were done")
    tooth_area: Optional[str] = Field(default=None, description="Tooth or area mentioned, if any")

class ExtractionResult(BaseModel):
    patient_name: Optional[str] = Field(default=None, description="Patient's name, null if not mentioned")
    procedures: List[ProcedureItem] = Field(default_factory=list)
    ambiguous: bool = Field(description="True if patient name is missing/unclear")
    clarification_needed: Optional[str] = Field(
        default=None, description="Short Arabic question to ask the doctor if something is unclear, else null"
    )

extractor = llm.with_structured_output(ExtractionResult)

EXTRACTION_PROMPT = """You are a dental procedure extractor for an Egyptian clinic.
The doctor describes procedures they just performed, in Egyptian Arabic dialect.
Extract the patient's name and each procedure with its quantity and tooth area if mentioned.
If the patient's name is missing, set ambiguous=true and ask for it in clarification_needed.
If a procedure name is too vague to price, set clarification_needed to a short Arabic question."""

print("Extractor ready.")

In [ ]:
def normalize_arabic(text: str, strip_al: bool = True) -> str:
    """Normalize Arabic text for matching: strip diacritics, unify hamza forms,
    optionally remove 'ال' (the) prefix from words, collapse extra spaces.
    FIX 10: strip_al is now optional — stripping 'ال' from patient NAMES mangled
    names that legitimately start with it (الاء -> اء, السيد -> سيد)."""
    text = re.sub(r"[\u064B-\u0652]", "", text)  # remove diacritics (tashkeel)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ة", "ه")
    if strip_al:
        text = re.sub(r"\bال", "", text)  # strip "ال" prefix from each word
    text = re.sub(r"\s+", " ", text).strip()
    return text


MIN_SUBSTRING_MATCH_LEN = 3  # FIX 10: a 1-2 char fragment must never match a whole record


def find_procedure(name: str):
    """FIX 10: exact-normalized match first; only fall back to substring matching,
    and only when the search term is long enough to be meaningful."""
    cur = conn.cursor()
    normalized_search = normalize_arabic(name)
    cur.execute("SELECT id, name, base_price, unit FROM procedures")
    rows = cur.fetchall()

    for row in rows:  # pass 1: exact
        if normalized_search == normalize_arabic(row[1]):
            return row

    if len(normalized_search) >= MIN_SUBSTRING_MATCH_LEN:  # pass 2: substring
        for row in rows:
            normalized_proc = normalize_arabic(row[1])
            if normalized_search in normalized_proc or normalized_proc in normalized_search:
                return row
    return None


def find_patient_by_name(name: str):
    """FIX 10: names keep their 'ال' (strip_al=False); exact matches win outright,
    substring fallback requires a minimum length."""
    cur = conn.cursor()
    normalized_search = normalize_arabic(name, strip_al=False)
    cur.execute("SELECT id, name, phone FROM patients")
    rows = cur.fetchall()

    exact = [r for r in rows if normalize_arabic(r[1], strip_al=False) == normalized_search]
    if exact:
        return exact

    if len(normalized_search) < MIN_SUBSTRING_MATCH_LEN:
        return []
    matches = []
    for row in rows:
        normalized_name = normalize_arabic(row[1], strip_al=False)
        if normalized_search in normalized_name or normalized_name in normalized_search:
            matches.append(row)
    return matches

print("Lookup helpers ready.")


In [ ]:
_pending = {}

CONFIRM_WORDS = ["تمام", "ايوه", "أيوه", "اه", "آه", "صح", "ok", "okay"]
DENY_WORDS = ["لأ", "لا", "غلط", "مش كده", "مش صح"]


def process_doctor_utterance(session_id: str, text: str) -> str:
    """Doctor says what they did -> extract -> price -> confirm -> write to DB
    (as a PENDING payment — money owed, not yet collected)."""
    text = text.strip()

    if session_id in _pending:
        pending = _pending[session_id]
        if any(w in text for w in CONFIRM_WORDS):
            cur = conn.cursor()
            # Dashboard shape: one pending payment + its procedure lines, exactly what
            # POST /api/doctor/charges writes -- the secretary's pending list picks this
            # up like any dashboard-posted charge. The ONE multi-statement write in the
            # notebook, so it gets an explicit transaction (created_at/performed_at
            # fall to the server's now() defaults).
            try:
                cur.execute("BEGIN")
                cur.execute(
                    """INSERT INTO payments (patient_id, base_amount, discount_amount,
                           insurance_covered_amount, final_amount, method, status, posted_by_doctor_id)
                       VALUES (%s, %s, 0, 0, %s, 'cash', 'pending', %s) RETURNING id""",
                    (pending["patient_id"], pending["total"], pending["total"], DOCTOR_USER_ID)
                )
                payment_id = cur.fetchone()[0]
                for item in pending["breakdown"]:
                    cur.execute(
                        """INSERT INTO patient_procedures (payment_id, patient_id, procedure_id,
                               quantity, price_charged, tooth_area)
                           VALUES (%s, %s, %s, %s, %s, %s)""",
                        (payment_id, pending["patient_id"], item["procedure_id"],
                         item["quantity"], item["line_total"], item["tooth_area"])
                    )
                cur.execute("COMMIT")
            except Exception:
                cur.execute("ROLLBACK")
                raise
            del _pending[session_id]
            return f"تمام، اتسجل. الإجمالي {pending['total']} جنيه (لسه مدفوعش)."
        elif any(w in text for w in DENY_WORDS):
            del _pending[session_id]
            return "تمام، قول تاني بالظبط عملت ايه ولمين."
        else:
            return "معلش مسمعتش، قول 'تمام' لو صح أو 'لأ' لو غلط."

    result = extractor.invoke(
        [{"role": "system", "content": EXTRACTION_PROMPT}, {"role": "user", "content": text}]
    )

    if result.ambiguous or not result.patient_name:
        return result.clarification_needed or "مين المريض اللي عملتله كده؟"

    matches = find_patient_by_name(result.patient_name)
    if len(matches) == 0:
        return f"مش لاقي مريض اسمه {result.patient_name} في السيستم. اتأكد من الاسم؟"
    if len(matches) > 1:
        names = "، ".join(f"{m[1]} ({m[2]})" for m in matches)
        return f"لاقيت أكتر من مريض بنفس الاسم: {names}. تقصد مين بالظبط؟"

    patient_id, patient_name, _ = matches[0]

    if not result.procedures:
        return result.clarification_needed or "عملت ايه بالظبط؟"

    breakdown = []
    total = 0
    for item in result.procedures:
        proc = find_procedure(item.name)
        if not proc:
            return f"مش عارف سعر '{item.name}' — ممكن تقول اسم الإجراء بشكل تاني؟"
        proc_id, proc_name, base_price, unit = proc
        line_total = round(float(base_price) * item.quantity, 2)  # FIX 14
        total = round(total + line_total, 2)                      # FIX 14
        breakdown.append({
            "procedure_id": proc_id,
            "name": proc_name,
            "quantity": item.quantity,
            "tooth_area": item.tooth_area,
            "line_total": line_total,
        })

    _pending[session_id] = {"patient_id": patient_id, "breakdown": breakdown, "total": total}

    lines = "، ".join(f"{b['name']} × {b['quantity']} = {int(b['line_total'])} جنيه" for b in breakdown)
    return f"يعني {patient_name}: {lines}. الإجمالي {int(total)} جنيه. تمام؟"

print("process_doctor_utterance ready.")


Test with a real patient name from your DB:

In [ ]:
print(process_doctor_utterance("test_doc_1", "عملت لمحمد سيد حشو وحشوتين"))

In [ ]:
print(process_doctor_utterance("test_doc_1", "تمام"))

### Secretary functions — see who owes money, mark it collected

In [ ]:
def list_pending_payments():
    cur = conn.cursor()
    cur.execute("""
        SELECT pay.id, p.name, p.phone, pay.final_amount, pay.created_at
        FROM payments pay
        JOIN patients p ON pay.patient_id = p.id
        WHERE pay.status IN ('pending', 'partial')
        ORDER BY pay.created_at ASC
    """)
    return cur.fetchall()


def mark_payment_paid(payment_id: int, method: str = "cash"):
    cur = conn.cursor()
    cur.execute("SELECT status FROM payments WHERE id = %s", (payment_id,))
    row = cur.fetchone()
    if not row:
        return "No payment found with this id."
    if row[0] == "paid":
        return "This payment was already marked as paid."
    cur.execute(
        "UPDATE payments SET status = 'paid', method = %s, paid_at = %s WHERE id = %s",
        (method, now_cairo(), payment_id)
    )
    return f"Payment {payment_id} marked as paid ({method})."

print("Secretary functions ready.")


In [ ]:
print("Pending payments right now:")
for row in list_pending_payments():
    print(row)

### Agent 3 — Monthly Analytics Engine

In [ ]:
def get_month_range(year: int, month: int):
    """Half-open, Cairo-aware month boundaries. String boundaries against timestamptz
    get cast in the session's UTC and shift the month edges 2-3h (the exact bug the
    old FIX 15 fought in SQLite, reborn in Postgres)."""
    start = datetime(year, month, 1, tzinfo=CAIRO)
    end = datetime(year + (1 if month == 12 else 0), 1 if month == 12 else month + 1, 1, tzinfo=CAIRO)
    return start, end


def generate_revenue_summary(year: int, month: int) -> dict:
    start, end = get_month_range(year, month)
    cur = conn.cursor()
    cur.execute("""
        SELECT COALESCE(SUM(final_amount), 0), COUNT(*)
        FROM payments
        WHERE status = 'paid' AND paid_at >= %s AND paid_at < %s
    """, (start, end))
    total, count = cur.fetchone()
    return {"month": f"{year}-{month:02d}", "total_revenue": round(float(total), 2), "paid_count": count}


def identify_top_procedures(year: int, month: int, limit: int = 5) -> list:
    start, end = get_month_range(year, month)
    cur = conn.cursor()
    cur.execute("""
        SELECT pr.name, COUNT(*) as cnt, SUM(pp.price_charged) as revenue
        FROM patient_procedures pp
        JOIN procedures pr ON pp.procedure_id = pr.id
        WHERE pp.performed_at >= %s AND pp.performed_at < %s
        GROUP BY pr.id
        ORDER BY revenue DESC
        LIMIT %s
    """, (start, end, limit))
    return [{"procedure": r[0], "count": r[1], "revenue": r[2]} for r in cur.fetchall()]


def get_procedure_revenue_breakdown(year: int, month: int) -> list:
    start, end = get_month_range(year, month)
    cur = conn.cursor()
    cur.execute("""
        SELECT pr.name, COUNT(*) as cnt, SUM(pp.price_charged) as revenue
        FROM patient_procedures pp
        JOIN procedures pr ON pp.procedure_id = pr.id
        WHERE pp.performed_at >= %s AND pp.performed_at < %s
        GROUP BY pr.id
        ORDER BY revenue DESC
    """, (start, end))
    return [{"procedure": r[0], "count": r[1], "revenue": r[2]} for r in cur.fetchall()]


def get_total_patients_this_month(year: int, month: int) -> int:
    start, end = get_month_range(year, month)
    cur = conn.cursor()
    cur.execute("""
        SELECT COUNT(DISTINCT patient_id) FROM (
            SELECT patient_id FROM appointments WHERE scheduled_at >= %s AND scheduled_at < %s AND status != 'cancelled'
            UNION
            SELECT patient_id FROM patient_procedures WHERE performed_at >= %s AND performed_at < %s
        ) AS t
    """, (start, end, start, end))
    return cur.fetchone()[0]


def flag_inactive_patients(months_threshold: int = 6) -> list:
    """FIX 13: only COMPLETED visits (status='done') count as activity — a patient
    who booked-and-no-showed 8 months ago, or who merely has a future booking,
    was previously counted as 'active'."""
    cutoff = datetime.now(CAIRO) - timedelta(days=30 * months_threshold)
    cur = conn.cursor()
    # PG can't reference SELECT aliases in HAVING -- repeat the aggregate.
    cur.execute("""
        SELECT p.id, p.name, p.phone, MAX(a.scheduled_at) as last_visit
        FROM patients p
        LEFT JOIN appointments a ON a.patient_id = p.id AND a.status = 'done'
        GROUP BY p.id, p.name, p.phone
        HAVING MAX(a.scheduled_at) IS NULL OR MAX(a.scheduled_at) < %s
    """, (cutoff,))
    return [{"patient_id": r[0], "name": r[1], "phone": r[2], "last_visit": r[3]} for r in cur.fetchall()]


def calculate_no_show_impact(year: int, month: int) -> dict:
    start, end = get_month_range(year, month)
    cur = conn.cursor()
    cur.execute("""
        SELECT COUNT(*) FROM appointments
        WHERE status = 'no_show' AND scheduled_at >= %s AND scheduled_at < %s
    """, (start, end))
    no_show_count = cur.fetchone()[0]
    cur.execute("SELECT AVG(final_amount) FROM payments WHERE status = 'paid'")
    avg_payment = float(cur.fetchone()[0] or 0)
    return {"no_show_count": no_show_count, "estimated_lost_revenue": round(no_show_count * avg_payment, 2)}


def get_new_vs_returning_ratio(year: int, month: int) -> dict:
    start, end = get_month_range(year, month)
    cur = conn.cursor()
    cur.execute("""
        SELECT p.id,
               (SELECT COUNT(*) FROM appointments a2 WHERE a2.patient_id = p.id AND a2.scheduled_at < %s) as prior_count
        FROM patients p
        JOIN appointments a ON a.patient_id = p.id
        WHERE a.scheduled_at >= %s AND a.scheduled_at < %s
        GROUP BY p.id
    """, (start, start, end))
    rows = cur.fetchall()
    new_count = sum(1 for r in rows if r[1] == 0)
    returning_count = sum(1 for r in rows if r[1] > 0)
    return {"new_patients": new_count, "returning_patients": returning_count}


def generate_monthly_insights(year: int, month: int) -> str:
    """Aggregates all analytics and asks the LLM to generate Arabic business insights."""
    revenue = generate_revenue_summary(year, month)
    procedure_breakdown = get_procedure_revenue_breakdown(year, month)
    total_patients = get_total_patients_this_month(year, month)
    inactive = flag_inactive_patients(months_threshold=6)
    no_shows = calculate_no_show_impact(year, month)
    ratio = get_new_vs_returning_ratio(year, month)

    data_summary = f"""
Revenue this month: {revenue['total_revenue']} جنيه from {revenue['paid_count']} payments.
Total distinct patients seen this month: {total_patients}.
Revenue breakdown by procedure: {procedure_breakdown}
Inactive patients (6+ months): {len(inactive)} patients.
No-shows this month: {no_shows['no_show_count']}, estimated lost revenue: {no_shows['estimated_lost_revenue']} جنيه.
New patients: {ratio['new_patients']}, returning patients: {ratio['returning_patients']}.
"""

    prompt = f"""أنت محلل بيانات لعيادة أسنان مصرية. بناءً على البيانات التالية، اكتب 3 إلى 5 ملاحظات عملية ومفيدة للدكتور، باللغة العربية المصرية، بأسلوب مباشر وواضح.

البيانات:
{data_summary}

اكتب الملاحظات كنقاط قصيرة، كل نقطة تركز على حاجة واحدة يقدر الدكتور ياخد قرار بناء عليها."""

    response = llm.invoke(prompt)
    return response.content

print("Agent 3 analytics functions ready.")


In [ ]:
from datetime import date as _date
today = _date.today()
print(generate_revenue_summary(today.year, today.month))
print(identify_top_procedures(today.year, today.month))

In [ ]:
print(generate_monthly_insights(today.year, today.month))

### Agent 4 — WhatsApp Booking Agent
**Fix explained:** `handle_whatsapp_message_v2` below is the piece that was missing — your `handle_incoming_openwa_message` already called it, but it didn't exist yet, which would have raised a `NameError` the moment a real message came in. It also always sends replies to the real `chat_id` (never rebuilds one from a phone number), and it self-heals `whatsapp_chat_id` on every turn so future notifications (confirmations, cancellations, reminders) reach the patient correctly too.

In [ ]:
class RouteDecision(BaseModel):
    route: Literal["direct", "agent"] = Field(
        description="'direct' ONLY for greetings, thanks, or pure small talk with zero actionable content. "
                    "'agent' for anything involving booking, rescheduling, cancelling, asking about appointments, or any real request — even simple ones."
    )
    complexity: Literal["simple", "complex"] = Field(
        description="'simple' if a single clear request the small model can handle alone. "
                    "'complex' if it bundles multiple requests, is contradictory, angry/confused, or otherwise needs stronger reasoning."
    )

router_structured = llm_mini.with_structured_output(RouteDecision, method="json_mode")

# FIX 16: cheap keyword escalation used mid-conversation instead of an LLM call per turn.
ESCALATION_KEYWORDS = [
    "زعلان", "مش عاجبني", "وحش", "سيء", "مش راضي", "هشتكي", "شكوى",
    "عايز حد", "عايزة حد", "بني ادم", "بني آدم", "انسان حقيقي", "إنسان حقيقي",
    "موظف", "مسؤول", "المدير",
]

def route_message(text: str) -> RouteDecision:
    """FIX 6: wrapped in try/except — v1 let a json_mode failure on the 20b model
    propagate up, which marked the message processed and silently dropped it
    (patient never got any reply). Now a router failure just escalates safely."""
    try:
        return router_structured.invoke(
            f"Classify this patient message for an Egyptian dental clinic assistant. "
            f"Respond with a JSON object matching the required schema.\n"
            f"route='direct' ONLY if it's a greeting, thanks, or pure small talk with no actionable request.\n"
            f"route='agent' for ANY booking, rescheduling, cancelling, question, or real request — even simple ones.\n\n"
            f"complexity='complex' if ANY of these apply:\n"
            f"- the message expresses anger, frustration, or complaint (e.g. 'مش عاجبني', 'زعلان', 'وحش')\n"
            f"- the patient explicitly asks for a human / real person instead of the bot\n"
            f"- the message bundles multiple distinct unrelated requests\n"
            f"- the message is contradictory or confusing to parse\n"
            f"Otherwise complexity='simple', even if it contains multiple pieces of info about ONE single request "
            f"(like a booking with name+date+time all at once — that is still 'simple').\n\n"
            f"Message: {text}"
        )
    except Exception as e:
        print(f"   ⚠️ Router failed ({type(e).__name__}) — defaulting to agent/complex.")
        return RouteDecision(route="agent", complexity="complex")

DIRECT_REPLIES_KEYWORDS_HINT = "شكراً! لو احتجت أي حاجة تانية أنا موجود. 😊"

print("Router ready.")


In [ ]:
class BookingExtraction(BaseModel):
    patient_name: Optional[str] = Field(default=None, description="Patient's name, if mentioned")
    date: Optional[str] = Field(default=None, description="Requested date in YYYY-MM-DD format if mentioned and unambiguous, else null")
    time: Optional[str] = Field(default=None, description="Requested time in HH:MM 24-hour format if mentioned, else null")
    doctor_preference: Optional[str] = Field(default=None, description="Doctor's name if mentioned, else null")
    intent: Literal["book", "reschedule", "cancel", "unclear"] = Field(description="What the patient wants to do")

extractor_structured = llm_mini.with_structured_output(BookingExtraction)

def extract_booking_info(text: str) -> BookingExtraction:
    today = date.today()  # FIX: was hardcoded to a fixed date before
    weekday_ar = AR_WEEKDAYS[today.weekday()]
    try:
        return extractor_structured.invoke(
            f"Extract booking details from this Egyptian Arabic patient message for a dental clinic.\n"
            f"Today is {today.isoformat()}, a {today.strftime('%A')} (يوم {weekday_ar}).\n"
            f"Resolve relative dates yourself: بكرة = tomorrow, بعد بكرة = +2 days, "
            f"'الأربع الجاي'/'next Wednesday' = the next occurrence of that weekday after today.\n"
            f"Partial dates (like '30-8' = day 30, month 8) get the current year — but if that "
            f"day-month has already passed this year, use next year instead. Never require a year.\n"
            f"If a date or time is not mentioned at all, leave it null — do not guess.\n\n"
            f"Message: {text}"
        )
    except Exception as e:
        print(f"   ⚠️ Extraction failed ({type(e).__name__}), returning empty extraction.")
        return BookingExtraction(intent="unclear")

print("extract_booking_info ready.")

In [ ]:
def handle_whatsapp_message_v2(chat_id: str, phone: str, message_text: str) -> str:
    """Routes an incoming WhatsApp message. FIX 18: max_concurrency=1 forces LangGraph's
    ToolNode to execute tool calls SEQUENTIALLY -- by default it runs parallel tool calls
    in a thread pool, and multiple threads hitting our single shared SQLite connection
    caused 'DatabaseError: another row available'."""
    thread_config = {
        "configurable": {"thread_id": f"whatsapp_{chat_id}"},
        "max_concurrency": 1,   # FIX 18
    }
    existing_state = graph.get_state(thread_config)
    is_first_message = not existing_state.values.get("messages")

    if is_first_message:
        route_result = route_message(message_text)
        if route_result.route == "direct":
            reply_text = DIRECT_REPLIES_KEYWORDS_HINT
            _send_whatsapp_text_to_chat(chat_id, reply_text)
            return reply_text
        use_full = route_result.complexity == "complex"

        extracted = extract_booking_info(message_text)

        # v4 FIX 21: look the patient up ourselves, deterministically, instead
        # of just handing the LLM a phone number and trusting it to call
        # lookup_patient with it. We've seen the model second-guess an
        # @lid-derived number that doesn't look like a real phone and ask the
        # patient for one anyway -- exactly what we don't want; the whole
        # point of WhatsApp booking is the patient never has to type/say
        # their number. This also recognizes a RETURNING patient even on a
        # brand new conversation thread (e.g. right after a kernel restart
        # wipes LangGraph's in-memory state, even though the DB is untouched).
        cur = conn.cursor()
        cur.execute(
            "SELECT id, name FROM patients WHERE phone = %s OR whatsapp_chat_id = %s",
            (phone, chat_id)
        )
        existing_patient = cur.fetchone()

        if existing_patient:
            patient_id, patient_name = existing_patient
            known_facts = [
                f"(معلومة نظام تلقائية: المريض ده معروف بالفعل عندنا — patient_id={patient_id}، "
                f"الاسم: {patient_name}. متسألوش عن اسمه أو رقمه، ولا تستخدم lookup_patient أو "
                f"register_patient تاني — استخدم patient_id={patient_id} ده على طول في أي تول محتاجه.)"
            ]
        else:
            known_facts = [
                f"(معلومة نظام تلقائية — متسألش المريض عنها أبداً: رقم واتساب المرسل هو {phone}. "
                f"استخدمه مباشرة كـ phone في lookup_patient أو register_patient من غير ما تطلب من "
                f"المريض يقول رقمه أو يأكده.)"
            ]

        if extracted.patient_name:
            known_facts.append(f"اسم المريض المذكور: {extracted.patient_name}")
        if extracted.date:
            known_facts.append(f"التاريخ المطلوب: {extracted.date}")
        if extracted.time:
            known_facts.append(f"الوقت المطلوب: {extracted.time}")
        if extracted.doctor_preference:
            known_facts.append(f"الدكتور المطلوب: {extracted.doctor_preference}")
        enriched_text = "\n".join(known_facts) + f"\n{message_text}"
    else:
        use_full = any(kw in message_text for kw in ESCALATION_KEYWORDS)
        enriched_text = message_text

    result = None
    try:
        result = graph.invoke(
            {"messages": [HumanMessage(content=enriched_text)], "use_full_model": use_full},
            thread_config
        )
        reply_text = result["messages"][-1].content
    except Exception as e:
        # FIX 19: transient errors (rate limits, network) re-raise so the poll loop
        # RETRIES the message next cycle instead of apologizing and dropping it.
        msg = str(e).lower()
        if "429" in msg or "rate limit" in msg or "timeout" in msg or "connection" in msg:
            raise
        print(f"   ⚠️ Agent crashed processing message from {chat_id}: {type(e).__name__}: {e}")
        reply_text = "معلش، حصلت مشكلة تقنية عندنا. ممكن تبعت طلبك تاني؟"

    # v3 FIX 20: link whatsapp_chat_id straight to whichever patient this turn
    # actually touched, read from lookup_patient/register_patient's own result
    # in THIS turn -- instead of only guessing by matching `phone` against
    # patients.phone. That guess silently fails whenever @lid resolution fails
    # (see _resolve_lid_to_real_jid): `phone` here then ends up being the LID's
    # own unresolvable digits, which never matches the patient's real row, so
    # whatsapp_chat_id never gets healed and every later WhatsApp send (booking
    # confirmation, cancellation, reschedule) falls back to a phone-based id
    # that may not even be reachable.
    if result is not None:
        for m in result["messages"]:
            if getattr(m, "name", None) in ("lookup_patient", "register_patient"):
                match = re.search(r"patient_id=(\d+)", str(m.content))
                if match:
                    cur = conn.cursor()
                    cur.execute(
                        """UPDATE patients SET whatsapp_chat_id = %s
                           WHERE id = %s AND (whatsapp_chat_id IS NULL OR whatsapp_chat_id != %s)""",
                        (chat_id, int(match.group(1)), chat_id)
                    )

    # Legacy fallback heal (kept as a safety net for cases the loop above
    # doesn't cover, e.g. no lookup/register tool call happened this turn).
    cur = conn.cursor()
    cur.execute(
        """UPDATE patients SET whatsapp_chat_id = %s
           WHERE (phone = %s OR whatsapp_chat_id = %s)
             AND (whatsapp_chat_id IS NULL OR whatsapp_chat_id != %s)""",
        (chat_id, phone, chat_id, chat_id)
    )

    sent = _send_whatsapp_text_to_chat(chat_id, reply_text)
    if not sent:
        print(f"WARNING: failed to send WhatsApp reply to {chat_id}")

    return reply_text

print("handle_whatsapp_message_v2 ready (sequential tools + transient retry + deterministic chat-id heal + deterministic patient lookup).")


Test with a plain text message first (no voice, no real WhatsApp send needed to verify the logic):

In [ ]:
# DISABLED: this test used a FAKE number (201066744255) -- not a real WhatsApp
# account, so the reply-send ALWAYS fails with a 500 and it registered a junk
# "anonymous" patient in the DB. It made the bot look broken when it wasn't.
# Test the real flow with the polling loop below + a real phone instead.
# print(handle_whatsapp_message_v2("201066744255@c.us", "201066744255", "شكراً جدا"))
print("skipped -- test with a real WhatsApp message via the polling loop instead.")


In [ ]:
# DISABLED: same reason as above -- 201099887766 is a fake number, the send can
# never succeed (the OpenWA log shows 'No LID for user' for it every time).
# print(handle_whatsapp_message_v2("201099887766@c.us", "201099887766", "عايز احجز معاد يوم 30-8 الساعة 10 الصبح"))
print("skipped -- test with a real WhatsApp message via the polling loop instead.")


### Voice note transcription (Groq Whisper large-v3)

In [ ]:
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))


def transcribe_audio(file_path: str) -> str:
    """Transcribe an audio file to Arabic text using Groq's Whisper large-v3.
    Uses the shared _with_groq_retry helper (defined right after the LLM cell,
    also used by agent()'s own LLM call -- see its docstring for why)."""
    def _call():
        with open(file_path, "rb") as f:
            return groq_client.audio.transcriptions.create(
                file=f,
                model="whisper-large-v3",
                language="ar",
                response_format="json",
                temperature=0.0
            )
    return _with_groq_retry(_call).text


def save_and_transcribe_voice(media_data_base64: str, mimetype: str) -> str:
    ext = "ogg" if "ogg" in mimetype else "m4a"
    audio_bytes = base64.b64decode(media_data_base64)
    with tempfile.NamedTemporaryFile(suffix=f".{ext}", delete=False) as f:
        f.write(audio_bytes)
        temp_path = f.name
    try:
        return transcribe_audio(temp_path)
    finally:
        try:
            os.unlink(temp_path)
        except OSError:
            pass

print("Transcription helpers ready (using shared 429 retry).")


In [ ]:
def get_patient_chat_history(phone: str = None, chat_id: str = None) -> str:
    """Pull the full conversation history for one patient from LangGraph's memory —
    works whether you know their phone number or their raw WhatsApp chat_id.
    Returns a readable transcript (patient turns + agent turns), or a message
    saying there's no history yet."""
    if not chat_id:
        if not phone:
            return "You must provide either phone or chat_id."
        cur = conn.cursor()
        cur.execute("SELECT whatsapp_chat_id FROM patients WHERE phone = %s", (phone,))
        row = cur.fetchone()
        if not row or not row[0]:
            chat_id = f"{phone}@c.us"  # best guess if they never messaged on WhatsApp yet
        else:
            chat_id = row[0]

    thread_config = {"configurable": {"thread_id": f"whatsapp_{chat_id}"}}
    state = graph.get_state(thread_config)
    messages = state.values.get("messages", [])

    if not messages:
        return f"No conversation history yet for {chat_id}."

    lines = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"👤 Patient: {m.content}")
        elif isinstance(m, SystemMessage):
            continue  # skip the system prompt, not part of the actual conversation
        else:
            # AIMessage — only show it if it has visible text (skip pure tool-call steps)
            if getattr(m, "content", None):
                lines.append(f"🤖 Agent: {m.content}")

    return "\n".join(lines) if lines else f"No visible conversation turns yet for {chat_id}."


print("get_patient_chat_history ready.")


In [ ]:
def get_message_text(msg: dict) -> str | None:
    """Takes a raw OpenWA message (text or voice) and returns plain text either way.
    Returns None if the message type isn't supported OR the voice media isn't
    available (so the caller can reply asking the patient to type instead)."""
    msg_type = msg.get("type")

    if msg_type == "voice":
        # FIX: OpenWA can return metadata as an explicit null (not just a missing
        # key), so `msg.get("metadata", {})` returns None and crashes on .get().
        # Also, when OpenWA's engine fails to download the audio from WhatsApp,
        # there is simply no media to transcribe -> degrade gracefully instead
        # of crashing the whole poll cycle.
        media = (msg.get("metadata") or {}).get("media") or {}
        if not media.get("data"):
            print(f"WARNING: voice message from {msg.get('from')} has no downloadable media "
                  f"(OpenWA couldn't fetch the audio) -- will ask them to type.")
            return None
        text = save_and_transcribe_voice(media["data"], media["mimetype"])
        print(f"   \U0001f399\ufe0f Transcribed voice from {msg.get('from')}: {text}")
        return text

    elif msg_type == "text":
        return msg.get("body", "")

    else:
        print(f"WARNING: unsupported message type '{msg_type}' from {msg.get('from')}")
        return None

print("get_message_text ready.")


In [ ]:
def handle_incoming_openwa_message(msg: dict) -> str | None:
    """Takes one raw OpenWA message (text or voice) and routes it through the
    booking agent. Each distinct WhatsApp chat_id gets its own conversation
    thread/memory automatically — handle_whatsapp_message_v2 keys the LangGraph
    checkpointer by f"whatsapp_{chat_id}", so patient A's conversation state
    never leaks into patient B's, even if their messages get processed back to back."""
    chat_id = msg["from"]          # real chat_id from WhatsApp — never rebuild it

    # FIX 4: for @lid contacts, the part before '@' is a privacy LID, NOT a phone
    # number. v1 fed that LID to lookup_patient (never found) and then registered a
    # DUPLICATE patient whose "phone" was the LID. Resolve the real number first.
    if chat_id.endswith("@lid"):
        resolved = _resolve_lid_to_real_jid(chat_id)
        phone = resolved.split("@")[0]
        if resolved == chat_id:
            print(f"   ⚠️ Could not resolve real phone for {chat_id} — "
                  f"the agent will treat '{phone}' as the phone; expect a possible duplicate patient.")
    else:
        phone = chat_id.split("@")[0]

    message_text = get_message_text(msg)
    if message_text is None:
        _send_whatsapp_text_to_chat(chat_id, "معلش، مقدرتش أفهم الرسالة دي. ممكن تكتبها أو تبعتها صوت؟")
        return None

    return handle_whatsapp_message_v2(chat_id, phone, message_text)

print("handle_incoming_openwa_message ready.")


In [ ]:
import time

MY_OWN_NUMBER = "201017989362"

_processed_message_ids = set()
_bot_started_at = None


def _ts_to_seconds(ts) -> float:
    """Handles numeric seconds, numeric milliseconds, and ISO strings like
    '2026-07-13T17:22:26.000Z' (which is what your OpenWA actually sends)."""
    if ts is None or ts == "":
        return 0.0
    if isinstance(ts, (int, float)):
        return ts / 1000.0 if ts > 1e12 else float(ts)
    if isinstance(ts, str):
        s = ts.strip()
        try:
            n = float(s)
            return n / 1000.0 if n > 1e12 else n
        except ValueError:
            pass
        try:
            return datetime.fromisoformat(s.replace("Z", "+00:00")).timestamp()
        except ValueError:
            print(f"   \u26a0\ufe0f Could not parse createdAt value {ts!r} -- treating as 0 (very old).")
            return 0.0
    return 0.0


def poll_and_process_new_messages(limit: int = 25):
    global _bot_started_at

    _ensure_conn()  # the Supabase pooler may have dropped an idle connection between patients

    resp = requests.get(
        f"{OPENWA_URL}/api/sessions/{OPENWA_SESSION_ID}/messages",
        headers={"X-API-Key": OPENWA_API_KEY},
        params={"includeMedia": "true", "limit": limit}
    )
    resp.raise_for_status()
    messages = resp.json()["messages"]
    messages_sorted = sorted(messages, key=lambda m: _ts_to_seconds(m.get("createdAt", 0)))

    if _bot_started_at is None:
        _bot_started_at = max(
            (_ts_to_seconds(m.get("createdAt", 0)) for m in messages_sorted),
            default=time.time()
        )
        for m in messages_sorted:
            _processed_message_ids.add(m["id"])
        print(f"Baseline set -- ignoring {len(messages_sorted)} existing messages.")
        return []

    new_incoming = [
        m for m in messages_sorted
        if not m.get("fromMe", False)
        and m["id"] not in _processed_message_ids
        and m.get("type") in ("text", "voice")
        and "@g.us" not in m.get("from", "")
        and "@newsletter" not in m.get("from", "")      # v5: channels post here too
        and "@broadcast" not in m.get("from", "")       # v5: status@broadcast etc.
        and MY_OWN_NUMBER not in m.get("from", "")
        and _ts_to_seconds(m.get("createdAt", 0)) > _bot_started_at
    ]

    results = []
    for msg in new_incoming:
        # Log the actual trigger before processing it, so it's never ambiguous
        # whether a reply corresponds to a real new incoming message.
        preview_in = (msg.get("body") or "")[:80] if msg.get("type") == "text" else "[voice note]"
        print(f"\U0001f4e9 New {msg['type']} message from {msg['from']}: {preview_in}")

        # No retry queue: process the message exactly once. If it fails, mark it
        # processed anyway (so we never re-loop on the same broken message),
        # print one concise line, and send a single best-effort apology.
        _processed_message_ids.add(msg["id"])
        try:
            reply = handle_incoming_openwa_message(msg)
            results.append((msg, reply))
            preview = (reply[:120] + "\u2026") if reply and len(reply) > 120 else reply
            print(f"\u2705 [{msg['type']}] from {msg['from']}: replied -> {preview}")
        except Exception as e:
            print(f"\u274c Error processing {msg['from']}: {type(e).__name__}: {e}")
            try:
                _send_whatsapp_text_to_chat(msg["from"], "\u0645\u0639\u0644\u0634\u060c \u062d\u0635\u0644\u062a \u0645\u0634\u0643\u0644\u0629 \u062a\u0642\u0646\u064a\u0629 \u0639\u0646\u062f\u0646\u0627. \u0645\u0645\u0643\u0646 \u062a\u0628\u0639\u062a \u0631\u0633\u0627\u0644\u062a\u0643 \u062a\u0627\u0646\u064a\u061f")
            except Exception:
                pass

    if not new_incoming:
        print("No new incoming messages.")
    return results

print("poll_and_process_new_messages ready (no retry queue).")


In [ ]:
poll_and_process_new_messages()

In [ ]:
def run_polling_loop(interval_seconds: int = 10):
    """Keeps calling poll_and_process_new_messages() forever, every interval_seconds.
    Still purely reactive — it only ever replies to a real incoming message, same
    guarantee as before, just automatic instead of you clicking run each time.
    Stop it anytime with the notebook's stop/interrupt button."""
    print(f"Polling every {interval_seconds}s. Interrupt the cell (stop button) to stop.")
    while True:
        try:
            poll_and_process_new_messages()
        except Exception as e:
            print(f"⚠️ Poll cycle failed, will retry next cycle: {e}")
        time.sleep(interval_seconds)

print("run_polling_loop ready.")

In [ ]:
run_polling_loop()